In [1]:
import pandas as pd
import numpy as numpy

In [3]:
import os
import re
from pathlib import Path

INPUT_DIRS = ["data2022", "data2023", "data2024", "data2025"]
OUTPUT_DIR = "snippets-amazon"
os.makedirs(OUTPUT_DIR, exist_ok=True)

jpmorgan_files = pd.read_csv("amazon_filings.csv")
valid_filenames = set(jpmorgan_files["File Name"].tolist())

# Define regex pattern to match EPS-related keywords
EPS_KEYWORDS = [ 
        # Forward-looking EPS guidance keywords and related forward-looking terms
        r'eps',
        r'eps guidance',
        r'earnings per share guidance',
        r'projected eps',
        r'expected eps',
        r'forecasted eps',
        r'guidance for eps',
        r'guidance for earnings per share',
        r'future eps',
        r'future earnings per share',
        r'outlook for eps',
        r'outlook for earnings per share',
        r'anticipated eps',
        r'anticipated earnings per share',
        r'next quarter eps guidance',
        r'next year eps guidance',
        r'forward-looking eps',
        r'forward-looking earnings per share',
        r'eps estimate',
        r'earnings per share estimate',
        r'eps target',
        r'earnings per share target',
        r'guidance.*eps',
        r'guidance.*earnings per share',
        r'projected.*eps',
        r'projected.*earnings per share',
        r'expected.*eps',
        r'expected.*earnings per share',
        r'forecast.*eps',
        r'forecast.*earnings per share',
        r'outlook.*eps',
        r'outlook.*earnings per share',
        r'anticipate.*eps',
        r'anticipate.*earnings per share',
        r'estimate.*eps',
        r'estimate.*earnings per share',
        r'target.*eps',
        r'target.*earnings per share'
    
]
keyword_pattern = re.compile('|'.join(EPS_KEYWORDS), re.IGNORECASE)

def get_word_window(text, keyword_regex, window=100):
    words = text.split()
    snippets = []

    for i, word in enumerate(words):
        joined = ' '.join(words[max(0, i - window): min(len(words), i + window + 1)])
        if keyword_regex.search(word):
            snippets.append(joined)

    # Remove exact duplicates while preserving order
    seen = set()
    unique_snippets = []
    for s in snippets:
        if s not in seen:
            seen.add(s)
            unique_snippets.append(s)

    return '\n'.join(unique_snippets)

# Process each .txt file in all input directories
for input_dir in INPUT_DIRS:
    for filepath in Path(input_dir).glob("*.txt"):
        if filepath.name not in valid_filenames:
            continue

        with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
            content = f.read()

        snippet_text = get_word_window(content, keyword_pattern)
        if snippet_text.strip():  # Save only if there's a match
            output_file = Path(OUTPUT_DIR) / filepath.name
            with open(output_file, "w", encoding="utf-8") as out:
                out.write(snippet_text)


In [ ]:
import openai
import pandas as pd
from pathlib import Path
import time

client = openai.OpenAI(api_key="API_KEY")
SNIPPETS_DIR = "snippets-amazon"

# Read the JPMorgan filings list
jpmorgan_files = pd.read_csv("amazon_filings.csv")
valid_filenames = set(jpmorgan_files["File Name"].tolist())

def extract_eps_from_text(text):
    prompt = f"""
You are a financial-analysis assistant.

You will receive a text snippet from an SEC 8-K filing.
Your job is to determine whether the company is issuing forward-looking quarterly EPS guidance indicating an increase, decrease, or no change in expected EPS.

Follow these strict instructions:

Only consider statements that clearly refer to a specific future quarter (e.g., “Q1,” “second quarter,” “Q125,” “Q225,” etc.).
– Accept phrasing like “expects,” “forecast,” “guidance,” “projects,” “outlook,” “revising,” “maintaining,” “reaffirming,” or similar future-oriented language.
– Ignore any EPS references to past quarters, the current quarter, or annual/multi-quarter periods.

You do not need to extract the EPS number unless it is explicitly mentioned.
– If a number like $1.25 or $1.25–$1.35 is mentioned as part of the future quarterly guidance, include it.
– Otherwise, just infer the trend direction.

Determine whether this guidance implies management expects EPS to:
• +1 – Increase
• -1 – Decrease
• 0 – Stay the same or direction not clear

Return your answer in this exact format:

<TREND>///<EPS or None>

Where:
• <EPS>: $X.XX, $X.XX–$Y.YY, or None
• <TREND>: +1, -1, or 0

Do not return any commentary or extra formatting.

Here is the input text:

{text}
    """.strip()

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=20
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"⚠️ Error on file: {e}")
        return "ERROR"

# Collect results
results = []

for snippet_file in sorted(Path(SNIPPETS_DIR).glob("*.txt")):
    if snippet_file.name not in valid_filenames:
        continue

    with open(snippet_file, "r", encoding="utf-8") as f:
        snippet = f.read().strip()

    if snippet:
        print(f"Processing {snippet_file.name}...")
        eps_value = extract_eps_from_text(snippet)
    else:
        eps_value = "EMPTY"

    results.append({
        "filename": snippet_file.name,
        "eps_extracted": eps_value
    })

    time.sleep(1)

# Save to CSV
df = pd.DataFrame(results)
df.to_csv("eps_extracted_results_amazon.csv", index=False)
print("✅ Done! Saved to eps_extracted_results_jpm.csv")


Processing 0001018724-22-000002.txt...
Processing 0001018724-22-000011.txt...
Processing 0001018724-22-000017.txt...
Processing 0001018724-23-000002.txt...
Processing 0001018724-23-000010.txt...
Processing 0001018724-24-000006.txt...
Processing 0001018724-24-000081.txt...
Processing 0001018724-25-000002.txt...
Processing 0001104659-24-057026.txt...
Processing 0001104659-25-033450.txt...
Processing 0001193125-22-104336.txt...
Processing 0001193125-23-003621.txt...
✅ Done! Saved to eps_extracted_results_jpm.csv


In [5]:
df

,filename,eps_extracted
0,0001018724-22-000002.txt,0///None
1,0001018724-22-000011.txt,0///None
2,0001018724-22-000017.txt,0///None
3,0001018724-23-000002.txt,0///None
4,0001018724-23-000010.txt,0///None
5,0001018724-24-000006.txt,0///None
6,0001018724-24-000081.txt,0///None
7,0001018724-25-000002.txt,0///None
8,0001104659-24-057026.txt,0///None
9,0001104659-25-033450.txt,0///None


In [18]:
df

,filename,eps_extracted
0,0000019617-25-000040.txt,None
1,0000019617-25-000042.txt,None///None
2,0000019617-25-000332.txt,None
3,0000019617-25-000334.txt,None
